# Lab. 23 - GANs

In [11]:
import torch
import os
import torch.nn as nn
import torch.utils.data as data
from torchvision import datasets, transforms, utils
import numpy as np
import math
import networkx as nx
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [12]:
transform = transforms.ToTensor()  # pixels in [0,1]
bs = 128

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=False)

# data loaders
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=bs, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=bs, shuffle=False)

# Create output directory if it does not exist
os.makedirs("_output", exist_ok=True)

## Ex. 1) MLP-Style GAN

In [39]:
class Generator(nn.Module):

    def __init__(self, n_input, n_output):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_input, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, n_output),
            nn.Tanh(),  # This maps values into [-1, 1]
        )

    def forward(self, x):
        return self.layers(x)


class Discriminator(nn.Module):

    def __init__(self, n_input):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_input, 1024),
            nn.LeakyReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),  # This normalizes the value
        )

    def forward(self, x):
        return self.layers(x)


LATENT_DIM = 100
IMAGE_DIM = 28 * 28
BATCH_SIZE = 128
EPOCHS = 10
LR = 2e-4

G_model = Generator(LATENT_DIM, IMAGE_DIM).to(device)
D_model = Discriminator(IMAGE_DIM).to(device)

G_optimizer = torch.optim.Adam(G_model.parameters(), lr=LR)
D_optimizer = torch.optim.Adam(D_model.parameters(), lr=LR)

loss_function = nn.BCELoss()  # This works for both actually!

for epoch in range(EPOCHS):
    current_G_loss = 0.0
    current_D_loss = 0.0

    G_model.train()
    D_model.train()
    for i, (real, _) in enumerate(train_loader):
        # Flatten the image input (preserving batch dimensions)
        real = real.view(-1, IMAGE_DIM).to(device)

        # Build labels
        current_batch_size = real.size(0)
        real_labels = torch.ones(current_batch_size, 1).to(device)
        fake_labels = torch.zeros(current_batch_size, 1).to(device)

        # First, train Discriminator

        # Generate as many fake examples as there are real ones
        z = torch.randn(current_batch_size, LATENT_DIM).to(device)
        fake = G_model(z)

        real_preds = D_model(real)
        fake_preds = D_model(fake)

        real_loss = loss_function(real_preds, real_labels)
        fake_loss = loss_function(fake_preds, fake_labels)
        D_loss = real_loss + fake_loss
        current_D_loss += D_loss.item()

        D_optimizer.zero_grad()
        D_loss.backward()
        D_optimizer.step()

        # Then, train Generator (with different data just cause it's easier to handle the two processes separatedly)
        z = torch.randn(current_batch_size, LATENT_DIM).to(device)
        fake = G_model(z)

        fake_preds = D_model(fake)

        G_loss = loss_function(fake_preds, real_labels) # NOTICE: real_labels!
        current_G_loss += G_loss.item()

        G_optimizer.zero_grad()
        G_loss.backward()
        G_optimizer.step()

    print(f"Epoch: {epoch:3d} | G Loss: {current_G_loss / len(train_loader)} | D Loss: {current_D_loss / len(train_loader)}")
    
with torch.no_grad():
    z = torch.randn(10, LATENT_DIM).to(device)
    generated = G_model(z).view(-1, 1, 28, 28)
    grid = utils.make_grid(generated, nrow=8, normalize=True)
    utils.save_image(grid, os.path.join("_output", f"generated_by_MLP.png"))
    

Epoch:   0 | G Loss: 2.4673376245094514 | D Loss: 0.8711411406252303
Epoch:   1 | G Loss: 2.3192247559012635 | D Loss: 0.6973891929824596
Epoch:   2 | G Loss: 2.992168180723943 | D Loss: 0.5635453756334685
Epoch:   3 | G Loss: 3.1614183672963936 | D Loss: 0.32092700545975905
Epoch:   4 | G Loss: 4.012019075564484 | D Loss: 0.3248466031510693
Epoch:   5 | G Loss: 4.220919060554586 | D Loss: 0.2592599567518369
Epoch:   6 | G Loss: 4.545061897875658 | D Loss: 0.2204482039845765
Epoch:   7 | G Loss: 4.947115102556468 | D Loss: 0.23131441969130592
Epoch:   8 | G Loss: 5.645988216786496 | D Loss: 0.18640451996362825
Epoch:   9 | G Loss: 5.679653718781624 | D Loss: 0.16777392247441544


## Ex. 2) Convolutional GAN (DCGAN)

In [40]:
class Generator(nn.Module):

    def __init__(self, ch_latent, ch_output):
        super().__init__()
        self.layers = nn.Sequential(
            nn.ConvTranspose2d(ch_latent, 256, kernel_size=7, stride=1, padding=0, bias=False), # [...x1x1] -> [...x7x7]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False), # [...x7x7] -> [...x14x14]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False), # [...x14x14] -> [...x28x28]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, ch_output, kernel_size=3, stride=1, padding=1, bias=False), # [...x28x28] -> [...x28x28]
            nn.Tanh(),  # This maps values into [-1, 1]
        )

    def forward(self, x):
        return self.layers(x)


class Discriminator(nn.Module):

    def __init__(self, ch_input):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(ch_input, 64, kernel_size=4, stride=2, padding=1, bias=False), # [...x28x28] -> [...x14x14]
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False), # [...x14x14] -> [...x7x7]
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False), # [...x7x7] -> [...x7x7]
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, kernel_size=7, stride=1, padding=0, bias=False), # [...x7x7] -> [...x1x1]
        )

    def forward(self, x):
        return self.layers(x).view(x.size(0)) # Flatten to a vector
    
    
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


LATENT_DIM = 100
IMAGE_DIM = 28 * 28
BATCH_SIZE = 128
EPOCHS = 25
LR = 2e-4

G_model = Generator(LATENT_DIM, 1).to(device)
D_model = Discriminator(1).to(device)
G_model.apply(weights_init)
D_model.apply(weights_init)

G_optimizer = torch.optim.Adam(G_model.parameters(), lr=LR, betas=(0.5, 0.999))
D_optimizer = torch.optim.Adam(D_model.parameters(), lr=LR, betas=(0.5, 0.999))

loss_function = nn.BCEWithLogitsLoss()  # This works for both actually!

for epoch in range(EPOCHS):
    current_G_loss = 0.0
    current_D_loss = 0.0
    
    D_real_prob = 0.0
    D_fake_prob = 0.0

    G_model.train()
    D_model.train()
    for i, (real, _) in enumerate(train_loader):
        real = real.to(device)

        # Build labels
        current_batch_size = real.size(0)
        real_labels = torch.ones(current_batch_size).to(device) * 0.9
        fake_labels = torch.zeros(current_batch_size).to(device)

        # First, train Discriminator

        # Generate as many fake examples as there are real ones
        z = torch.randn(current_batch_size, LATENT_DIM, 1, 1).to(device)
        fake = G_model(z).detach() # Notice this detach

        real_preds = D_model(real)
        fake_preds = D_model(fake)

        real_loss = loss_function(real_preds, real_labels)
        fake_loss = loss_function(fake_preds, fake_labels)
        D_loss = real_loss + fake_loss
        current_D_loss += D_loss.item()

        D_optimizer.zero_grad()
        D_loss.backward()
        D_optimizer.step()

        with torch.no_grad():
            D_real_prob += torch.sigmoid(real_preds).mean().item()
            D_fake_prob += torch.sigmoid(fake_preds).mean().item()

        # Then, train Generator (with different data just cause it's easier to handle the two processes separatedly)
        for _ in range(3):
            z = torch.randn(current_batch_size, LATENT_DIM, 1, 1).to(device)
            fake = G_model(z)

            fake_preds = D_model(fake)

            G_loss = loss_function(fake_preds, real_labels) # NOTICE: real_labels!
            current_G_loss += G_loss.item()

            G_optimizer.zero_grad()
            G_loss.backward()
            G_optimizer.step()
        

    print(f"Epoch: {epoch:3d} | G Loss: {current_G_loss / len(train_loader)} | D Loss: {current_D_loss / len(train_loader)} | D(x) {D_real_prob / len(train_loader)} | D(G(z)) {D_fake_prob / len(train_loader)}")
    
with torch.no_grad():
    z = torch.randn(10, LATENT_DIM, 1, 1).to(device)
    generated = G_model(z)
    grid = utils.make_grid(generated, nrow=8, normalize=True)
    utils.save_image(grid, os.path.join("_output", f"generated_by_DCGAN.png"))
    

Epoch:   0 | G Loss: 7.3772175638025 | D Loss: 0.7013302454943342 | D(x) 0.7509852630306663 | D(G(z)) 0.1539522330410707
Epoch:   1 | G Loss: 6.936343443546214 | D Loss: 0.6680247127882707 | D(x) 0.75492731235556 | D(G(z)) 0.14524870280222074
Epoch:   2 | G Loss: 5.491080824246031 | D Loss: 0.8532506523610178 | D(x) 0.6820933036903328 | D(G(z)) 0.2181496217624465
Epoch:   3 | G Loss: 4.536978843687439 | D Loss: 0.9945236899451152 | D(x) 0.6248751499378351 | D(G(z)) 0.2749760292454569
Epoch:   4 | G Loss: 4.162868393001272 | D Loss: 1.0410113808696966 | D(x) 0.6031048504401372 | D(G(z)) 0.2963289386872798
Epoch:   5 | G Loss: 4.079627829319887 | D Loss: 1.0571914902373927 | D(x) 0.5969002999222355 | D(G(z)) 0.30359815454273337
Epoch:   6 | G Loss: 3.959629696251741 | D Loss: 1.071390423312116 | D(x) 0.5896507426619784 | D(G(z)) 0.3097658129111091
Epoch:   7 | G Loss: 3.8464440310687653 | D Loss: 1.1010459492455666 | D(x) 0.5796507685296317 | D(G(z)) 0.3203931822260814
Epoch:   8 | G Los

## Ex. 3) CycleGAN

This is a **teaching-friendly** CycleGAN skeleton for **unpaired image-to-image translation**:

- Two domains: $A$ and $B$ (e.g., horses ↔ zebras)
- Two generators: $G_{A\to B}$ and $G_{B\to A}$
- Two discriminators: $D_A$ and $D_B$ (PatchGAN)
- Losses:
  - Adversarial (LSGAN): makes outputs look real in the target domain
  - Cycle-consistency: $A \to B \to A$ reconstructs the original
  - Identity (optional): preserves color/structure when input already in target domain

> NOTE: This is not a full production CycleGAN (no buffer/replay, no schedulers, no mixed precision). It’s meant to be **simple and readable**.

In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from torchvision.datasets import CIFAR10
import random

@dataclass
class CFG:
    # If using CIFAR classes rather than folders, provide dataset root
    use_cifar: bool = True
    cifar_root: str = "./data"
    classA: int = 0    # airplane
    classB: int = 1    # automobile

    out_dir: str = "./_cyclegan_out_cifar"
    image_size: int = 32
    batch_size: int = 16           # can increase if GPU memory allows
    num_workers: int = 4

    lr: float = 2e-4
    beta1: float = 0.5
    epochs: int = 200

    lambda_cyc: float = 10.0
    lambda_id: float = 5.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    sample_every: int = 500


class CIFARUnpairedDataset(torch.utils.data.Dataset):
    """
    Build two pools from CIFAR-10 classes classA and classB.
    Returns (imgA, imgB) as tensors transformed to [-1,1].
    """
    def __init__(self, root, classA, classB, transform, train=True, download=True):
        self.transform = transform
        ds = CIFAR10(root=root, train=train, download=download)
        self.imagesA = [ds.data[i] for i, lbl in enumerate(ds.targets) if lbl == classA]
        self.imagesB = [ds.data[i] for i, lbl in enumerate(ds.targets) if lbl == classB]
        if len(self.imagesA) == 0 or len(self.imagesB) == 0:
            raise ValueError("No images for chosen classes")
    def __len__(self):
        return max(len(self.imagesA), len(self.imagesB))
    def __getitem__(self, idx):
        a = Image.fromarray(self.imagesA[idx % len(self.imagesA)])
        b = Image.fromarray(random.choice(self.imagesB))
        return self.transform(a), self.transform(b)

# ----------------------------
# Data (unpaired)
# ----------------------------
class UnpairedImageDataset(Dataset):
    """
    Returns an unpaired sample (A_i, B_j).
    Domain folders should contain images.
    """
    def __init__(self, rootA, rootB, transform):
        self.pathsA = sorted([str(p) for p in Path(rootA).glob("*") if p.is_file()])
        self.pathsB = sorted([str(p) for p in Path(rootB).glob("*") if p.is_file()])
        if len(self.pathsA) == 0 or len(self.pathsB) == 0:
            raise ValueError("Domain folders must contain images.")
        self.t = transform

    def __len__(self):
        return max(len(self.pathsA), len(self.pathsB))

    def __getitem__(self, idx):
        pathA = self.pathsA[idx % len(self.pathsA)]
        pathB = self.pathsB[torch.randint(0, len(self.pathsB), (1,)).item()]  # random B

        imgA = Image.open(pathA).convert("RGB")
        imgB = Image.open(pathB).convert("RGB")

        return self.t(imgA), self.t(imgB)


# ----------------------------
# Model blocks
# ----------------------------
class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),

            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class ResnetGenerator(nn.Module):
    """
    ResNet-based generator used in CycleGAN.
    Input/Output: (N, 3, H, W), output in [-1, 1] via Tanh.
    """
    def __init__(self, in_ch=3, out_ch=3, n_filters=64, n_blocks=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, n_filters, 7),
            nn.InstanceNorm2d(n_filters),
            nn.ReLU(True),
        ]

        # Downsample
        c = n_filters
        for _ in range(2):
            layers += [
                nn.Conv2d(c, c * 2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(c * 2),
                nn.ReLU(True),
            ]
            c *= 2

        # ResNet blocks
        for _ in range(n_blocks):
            layers += [ResnetBlock(c)]

        # Upsample
        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(c, c // 2, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(c // 2),
                nn.ReLU(True),
            ]
            c //= 2

        # Output
        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(c, out_ch, 7),
            nn.Tanh(),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class PatchDiscriminator(nn.Module):
    """
    PatchGAN discriminator.
    Outputs a map of logits (N, 1, H', W') — we use LSGAN (MSE) loss.
    """
    def __init__(self, in_ch=3, n_filters=64):
        super().__init__()
        def block(in_c, out_c, stride):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1),
                nn.InstanceNorm2d(out_c),
                nn.LeakyReLU(0.2, inplace=True),
            )

        layers = [
            nn.Conv2d(in_ch, n_filters, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            block(n_filters, n_filters * 2, 2),
            block(n_filters * 2, n_filters * 4, 2),
            block(n_filters * 4, n_filters * 8, 1),

            nn.Conv2d(n_filters * 8, 1, 4, stride=1, padding=1),  # logits map
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ----------------------------
# Loss helpers
# ----------------------------
def lsgan_loss(pred, target_is_real: bool):
    """
    LSGAN: MSE between discriminator logits-map and target labels (1 or 0).
    """
    target_val = 1.0 if target_is_real else 0.0
    target = torch.full_like(pred, fill_value=target_val)
    return torch.mean((pred - target) ** 2)


# ----------------------------
# Train
# ----------------------------
def main(cfg: CFG):
    torch.manual_seed(cfg.seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    # Transform: resize/crop + normalize to [-1,1]
    t = transforms.Compose([
        transforms.Resize(cfg.image_size),
        transforms.CenterCrop(cfg.image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),
                             (0.5, 0.5, 0.5)),
    ])

        # With this block:
    if cfg.use_cifar:
        ds = CIFARUnpairedDataset(
            root=cfg.cifar_root,
            classA=cfg.classA,
            classB=cfg.classB,
            transform=t,
            train=True,
            download=True,
        )
    else:
        ds = UnpairedImageDataset(cfg.rootA, cfg.rootB, t)
        
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, pin_memory=True)

    # Models
    G_AB = ResnetGenerator(n_blocks=9).to(cfg.device)
    G_BA = ResnetGenerator(n_blocks=9).to(cfg.device)
    D_A = PatchDiscriminator().to(cfg.device)
    D_B = PatchDiscriminator().to(cfg.device)

    # Optims
    opt_G = optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()),
                       lr=cfg.lr, betas=(cfg.beta1, 0.999))
    opt_D = optim.Adam(list(D_A.parameters()) + list(D_B.parameters()),
                       lr=cfg.lr, betas=(cfg.beta1, 0.999))

    l1 = nn.L1Loss()

    step = 0
    for epoch in range(cfg.epochs):
        for realA, realB in dl:
            realA = realA.to(cfg.device, non_blocking=True)
            realB = realB.to(cfg.device, non_blocking=True)

            # -----------------------------------------
            # (1) Train Generators: G_AB and G_BA
            # -----------------------------------------
            fakeB = G_AB(realA)
            fakeA = G_BA(realB)

            recA = G_BA(fakeB)  # A -> B -> A
            recB = G_AB(fakeA)  # B -> A -> B

            # GAN losses (LSGAN)
            loss_GAN_AB = lsgan_loss(D_B(fakeB), True)
            loss_GAN_BA = lsgan_loss(D_A(fakeA), True)

            # Cycle-consistency
            loss_cyc = l1(recA, realA) + l1(recB, realB)

            # Identity loss (optional)
            if cfg.lambda_id > 0:
                idA = G_BA(realA)  # should be ~realA
                idB = G_AB(realB)  # should be ~realB
                loss_id = l1(idA, realA) + l1(idB, realB)
            else:
                loss_id = torch.tensor(0.0, device=cfg.device)

            loss_G = (loss_GAN_AB + loss_GAN_BA) \
                     + cfg.lambda_cyc * loss_cyc \
                     + cfg.lambda_id * loss_id

            opt_G.zero_grad(set_to_none=True)
            loss_G.backward()
            opt_G.step()

            # -----------------------------------------
            # (2) Train Discriminators: D_A and D_B
            # -----------------------------------------
            with torch.no_grad():
                fakeB_det = fakeB.detach()
                fakeA_det = fakeA.detach()

            # D_A: realA vs fakeA
            loss_DA = 0.5 * (lsgan_loss(D_A(realA), True) + lsgan_loss(D_A(fakeA_det), False))
            # D_B: realB vs fakeB
            loss_DB = 0.5 * (lsgan_loss(D_B(realB), True) + lsgan_loss(D_B(fakeB_det), False))

            loss_D = loss_DA + loss_DB

            opt_D.zero_grad(set_to_none=True)
            loss_D.backward()
            opt_D.step()

            if step % 50 == 0:
                print(
                    f"epoch {epoch+1}/{cfg.epochs} step {step:06d} | "
                    f"lossG {loss_G.item():.3f} (gan {loss_GAN_AB.item()+loss_GAN_BA.item():.3f}, "
                    f"cyc {loss_cyc.item():.3f}, id {loss_id.item():.3f}) | "
                    f"lossD {loss_D.item():.3f}"
                )

            if step % cfg.sample_every == 0:
                # Save a small visual: realA, fakeB, recA and realB, fakeA, recB
                with torch.no_grad():
                    grid = torch.cat([realA[:1], fakeB[:1], recA[:1],
                                      realB[:1], fakeA[:1], recB[:1]], dim=0)
                    # unnormalize for saving: torchvision save_image handles normalize=True nicely
                    save_image(grid, os.path.join(cfg.out_dir, f"sample_{step:06d}.png"),
                               nrow=3, normalize=True, value_range=(-1, 1))

            step += 1

    print("Training complete.")
    
main(CFG())

100%|██████████| 170M/170M [00:46<00:00, 3.69MB/s] 
